# Ch.5 — Hyperparameter Tuning for Classification

**FaceAI**: Systematic search to push Smiling accuracy from 89% to ~92%.

**Methods**: Grid Search, Random Search, Bayesian Optimization (Optuna).

**Key unlock**: Per-attribute threshold tuning + `class_weight` for imbalanced attributes.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Import numpy, matplotlib.pyplot, Path
# 2. Import warnings; call warnings.filterwarnings('ignore')
# 3. Import make_classification, train_test_split, GridSearchCV,
#    RandomizedSearchCV, cross_val_score, StratifiedKFold
# 4. Import StandardScaler, SVC, LogisticRegression
# 5. Import f1_score, roc_auc_score, classification_report,
#    precision_recall_curve, make_scorer from sklearn.metrics
# 6. Import loguniform, uniform from scipy.stats
# 7. Set IMG_DIR, SAVE_KW, np.random.seed(42)
#
# Hint:
#   from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
#   from scipy.stats import loguniform, uniform
#   import warnings; warnings.filterwarnings('ignore')


## §0 Data — CelebA Smiling + Bald

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Generate X_smile, y_smile with make_classification(weights=[0.52, 0.48])
# 2. Generate X_bald, y_bald with make_classification(weights=[0.975, 0.025])
# 3. Split only the Smiling data for main training/test sets
# 4. Scale with StandardScaler; print shapes and Bald positive rate
#
# Hint:
#   X_smile, y_smile = make_classification(n_samples=5000, n_features=200,
#                                          n_informative=40, n_redundant=20,
#                                          n_clusters_per_class=3,
#                                          weights=[0.52, 0.48], flip_y=0.05,
#                                          random_state=42)
#   X_bald, y_bald   = make_classification(n_samples=5000, n_features=200,
#                                          n_informative=30, n_redundant=20,
#                                          weights=[0.975, 0.025], flip_y=0.03,
#                                          random_state=42)
#   X_tr, X_te, y_tr, y_te = train_test_split(X_smile, y_smile,
#                                              test_size=0.2, stratify=???, random_state=42)


## §1 Grid Search — Logistic Regression

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Define param_grid_lr:
#    'C': [0.01, 0.1, 1.0, 10.0, 100.0]
#    'penalty': ['l1', 'l2']
#    'class_weight': [None, 'balanced']
# 2. Create GridSearchCV(LogisticRegression(solver='saga', max_iter=500, random_state=42),
#    param_grid_lr, scoring='f1_macro', cv=5, n_jobs=-1, verbose=0)
# 3. Call grid_lr.fit(X_tr_s, y_tr)
# 4. Print best_params_, best_score_, test accuracy, and test F1
#
# Hint:
#   param_grid_lr = {
#       'C': [0.01, 0.1, 1.0, 10.0, 100.0],
#       'penalty': ['l1', 'l2'],
#       'class_weight': [None, 'balanced'],
#   }
#   grid_lr = GridSearchCV(
#       LogisticRegression(solver='saga', max_iter=500, random_state=42),
#       param_grid_lr, scoring='f1_macro', cv=5, n_jobs=-1)
#   grid_lr.fit(???, ???)


## §2 Random Search — SVM

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Define param_dist_svm using loguniform for 'C' (0.1, 100) and
#    'gamma' (1e-4, 1), plus lists for 'kernel' and 'class_weight'
# 2. Create RandomizedSearchCV(SVC(probability=True, random_state=42),
#    param_dist_svm, n_iter=30, scoring='f1_macro', cv=3, n_jobs=-1, random_state=42)
# 3. Fit and print best_params_, best_score_, and test accuracy
#
# Hint:
#   param_dist_svm = {
#       'C':            loguniform(0.1, 100),
#       'gamma':        loguniform(1e-4, 1),
#       'kernel':       ['rbf', 'linear'],
#       'class_weight': [None, 'balanced'],
#   }
#   random_svm = RandomizedSearchCV(
#       SVC(probability=True, random_state=42), param_dist_svm,
#       n_iter=???, scoring='f1_macro', cv=3, n_jobs=-1, random_state=42)
#   random_svm.fit(???, ???)


## §3 Bayesian Optimization (Optuna)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Import optuna; suppress logging with optuna.logging.set_verbosity(WARNING)
# 2. Define objective(trial) that:
#    a. Samples C via trial.suggest_float('C', 0.01, 100, log=True)
#    b. Samples gamma via trial.suggest_float('gamma', 1e-4, 1.0, log=True)
#    c. Samples class_weight via trial.suggest_categorical(...)
#    d. Returns cross_val_score mean with scoring='f1_macro', cv=3
# 3. Create study = optuna.create_study(direction='maximize')
# 4. Call study.optimize(objective, n_trials=40)
# 5. Train best_svm from study.best_params; print test accuracy
#    (wrap in try/except ImportError to handle missing optuna gracefully)
#
# Hint:
#   def objective(trial):
#       C     = trial.suggest_float('C', 0.01, 100, log=True)
#       gamma = trial.suggest_float('gamma', ???, ???, log=True)
#       cw    = trial.suggest_categorical('class_weight', [None, 'balanced'])
#       model = SVC(C=C, gamma=gamma, kernel='rbf', class_weight=cw, random_state=42)
#       return cross_val_score(model, X_tr_s, y_tr, cv=3, scoring='f1_macro').mean()
#   study = optuna.create_study(direction='maximize')
#   study.optimize(objective, n_trials=???)


## §4 Search Strategy Comparison

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Collect best CV F1-macro from grid_lr.best_score_, random_svm.best_score_,
#    and study.best_value (fallback to random_svm if optuna unavailable)
# 2. Create a bar chart comparing Grid, Random, and Bayesian strategies
# 3. Annotate each bar with its score; set ylim=(0.8, 1.0)
# 4. Save to IMG_DIR / 'search_comparison.png'
#
# Hint:
#   methods   = ['Grid (LogReg)', 'Random (SVM)', 'Bayesian (SVM)']
#   scores_cv = [grid_lr.best_score_, random_svm.best_score_,
#                study.best_value if 'study' in dir() else random_svm.best_score_]
#   bars = ax.bar(methods, scores_cv, color=['#2196F3', '#4CAF50', '#FF9800'])
#   for bar, score in zip(bars, scores_cv):
#       ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
#               f'{score:.4f}', ha='center', va='bottom')


## §5 Threshold Tuning

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Get y_prob_smile = best_svm.predict_proba(X_te_s)[:, 1]
# 2. Sweep thresholds = np.arange(0.1, 0.9, 0.02); compute F1 at each → best_t_smile
# 3. For Bald: split + scale, fit SVC(C=10, gamma=0.01, class_weight='balanced'),
#    sweep thresholds → best_t_bald
# 4. Create 1x2 subplots; mark optimal t and default 0.5 on each
# 5. Save to IMG_DIR / 'threshold_tuning.png'
#
# Hint:
#   y_prob_smile = best_svm.predict_proba(???)[: , 1]
#   f1s  = [f1_score(y_te, (y_prob_smile >= t).astype(int)) for t in thresholds]
#   best_t_smile = thresholds[np.argmax(f1s)]
#   svm_bald = SVC(C=10, gamma=0.01, kernel='rbf',
#                  class_weight='balanced', probability=True, random_state=42)


## §6 Nested Cross-Validation

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Create outer_cv = StratifiedKFold(n_splits=5) and
#    inner_cv = StratifiedKFold(n_splits=3), both with shuffle=True
# 2. Define param_grid = {'C': [0.1, 1, 10], 'gamma': [0.001, 0.01, 0.1]}
# 3. Loop over outer_cv.split(X_tr_s, y_tr):
#    a. Run GridSearchCV(SVC(kernel='rbf'), param_grid, cv=inner_cv) on inner train
#    b. Evaluate on outer test; compute f1_score(average='macro')
# 4. Print per-fold results and final mean ± std
#
# Hint:
#   outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#   inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
#   for i, (train_idx, test_idx) in enumerate(outer_cv.split(X_tr_s, y_tr)):
#       inner_grid = GridSearchCV(SVC(kernel='rbf', random_state=42),
#                                 param_grid, scoring='f1_macro', cv=inner_cv, n_jobs=-1)
#       inner_grid.fit(X_inner, y_inner)
#       score = f1_score(y_outer, inner_grid.predict(X_outer), average='macro')


## §7 Optuna History Visualization

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Check if 'study' is defined (Optuna available)
# 2. Extract trial_values = [t.value for t in study.trials]
# 3. Compute best_so_far = np.maximum.accumulate(trial_values)
# 4. Scatter plot all trial scores; overlay best-so-far as red line
# 5. Save to IMG_DIR / 'optuna_history.png'
#    (if study not available print a skip message instead)
#
# Hint:
#   if 'study' in dir():
#       trials       = study.trials
#       trial_values = [t.value for t in trials]
#       best_so_far  = np.maximum.accumulate(trial_values)
#       ax.scatter(range(len(trial_values)), trial_values, alpha=0.4, label='Trial score')
#       ax.plot(range(len(best_so_far)), best_so_far, 'r-', label='Best so far')


## §8 Final Model Performance

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Apply best threshold: y_pred_tuned = (y_prob_smile >= best_t_smile).astype(int)
# 2. Print classification_report(y_te, y_pred_tuned, target_names=[...])
# 3. Compute acc_tuned = (y_pred_tuned == y_te).mean()
# 4. Compute f1_tuned = f1_score(y_te, y_pred_tuned)
# 5. Compute auc = roc_auc_score(y_te, y_prob_smile); print all metrics
#
# Hint:
#   y_pred_tuned = (y_prob_smile >= best_t_smile).astype(int)
#   print(classification_report(y_te, y_pred_tuned,
#                               target_names=['Not Smiling', 'Smiling']))
#   auc = roc_auc_score(y_te, y_prob_smile)


## §9 Summary

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Print separator line and chapter title
# 2. Print accuracy progression: Ch.1 (~88%) -> Ch.4 (~89%) -> Ch.5 (tuned)
# 3. Print key unlocks: Grid/Random/Bayesian search, per-attribute threshold tuning
#
# Hint:
#   print(f"  Ch.1 LogReg baseline:  ~88%")
#   print(f"  Ch.4 SVM hand-tuned:   ~89%")
#   print(f"  Ch.5 Tuned model:      ~{acc_tuned*100:.0f}% ✅")
#   print("Key unlocks: systematic search + per-attribute threshold optimisation")


## Exercises

1. **Multi-attribute tuning**: Tune threshold for 5 attributes (Smiling, Bald, Eyeglasses, Male, Young) independently. Plot optimal thresholds.
2. **Search budget**: Compare Random Search with 10, 30, 100 iterations. At what point do diminishing returns kick in?
3. **Scoring function**: Repeat Grid Search with `scoring='roc_auc'` instead of `'f1_macro'`. Does the best model change?

In [ ]:
# Exercise 1: Multi-attribute threshold tuning
# TODO: Implement this cell
#
# Hint: create 5 synthetic attribute datasets with different positive rates;
#   for each fit best_svm and sweep thresholds to find the optimal t per attribute;
#   plot optimal thresholds as a bar chart to compare across attributes


In [ ]:
# Exercise 2: Search budget analysis
# TODO: Implement this cell
#
# Hint: run RandomizedSearchCV with n_iter in [10, 30, 100]; record best_score_
#   and wall-clock time (time.time()); plot score vs n_iter to show diminishing returns


In [ ]:
# Exercise 3: Scoring function comparison
# TODO: Implement this cell
#
# Hint: repeat GridSearchCV with same param_grid_lr but scoring='roc_auc';
#   compare best_params_ for both runs; print test AUC and F1 for each winner
